In [ ]:
# Gets all day directories (like D01, D02, etc.) for a specific subject.
# Filters folder names that begin with "D", sorts them, and returns the list of paths.
def get_day_paths(subject_path: Path): # checked
    days = os.listdir(subject_path)
    day_paths = sorted(list(map(lambda x: subject_path / x, list(filter(lambda x: x[0] == "D", days)))))

    return day_paths

In [ ]:
# Reads the interpolated.csv file inside the given day/segment folder 
# Returns a dictionary of DataFrames with keys: "left finger", "right finger", "left wrist", "right wrist"
def read_interpolated_sensor_csv(day_path: Path): # checked
    """
    Read all sensor data. Concatenate and order if there are more than one file.
    """
    left_right: pd.DataFrame = pd.read_csv(list(day_path.glob("interpolated.csv"))[0])

    left_finger: pd.DataFrame = left_right["left finger"]
    right_finger: pd.DataFrame = left_right["right finger"]
    left_wrist: pd.DataFrame = left_right["left wrist"]
    right_wrist: pd.DataFrame = left_right["right wrist"]

    return {"left finger": left_finger,
            "right finger": right_finger,
            "left wrist": left_wrist,
            "right wrist": right_wrist}

In [ ]:
# Gets all subject directories inside the main data folder, ignoring hidden files
# For each subject folder:
    # Calls get_day_paths() to get all day paths.
    # Stores them in a dictionary where the key is the subject name and the value is a list of day paths.
def get_subject_day_paths(data_path: Path) -> dict[str, list[Path]]: # checked
    all_subject_dir_names: list[str] = os.listdir(data_path)
    all_subject_paths: list[Path] = sorted(list(
        map(lambda x: data_path / x, list(filter(lambda x: x[0] != ".", all_subject_dir_names)))
    ))
    subject_day_paths = {}

    for subject_path in all_subject_paths:
        subject = subject_path.name
        subject_day_paths[subject] = get_day_paths(subject_path=subject_path)

    return subject_day_paths

In [ ]:
# Prepares the output folder structure to store assigned files.
def create_dirs(valid_subject_day_paths: dict[str, list[Path]], output_dir: Path): 
    subjects: list[str] = valid_subject_day_paths.keys()

    for subject in subjects:
        subject_output_path: Path = output_dir / subject
        if not os.path.exists(subject_output_path):
            os.makedirs(subject_output_path)

        day_paths: list[Path] = valid_subject_day_paths[subject]
        days: list[str] = list(map(lambda day_path: day_path.name[:6], day_paths))

        for day, day_path in zip(days, day_paths):
            day_output_path: Path = output_dir / subject / day
            if not os.path.exists(day_output_path):
                os.makedirs(day_output_path)

            files_in_day_path: list[str] = os.listdir(day_path)
            if "segment1" in files_in_day_path and not os.path.exists(day_output_path / "segment1"):
                os.makedirs(day_output_path / "segment1")

            if "segment2" in files_in_day_path and not os.path.exists(day_output_path / "segment2"):
                os.makedirs(day_output_path / "segment2")

            if "segment3" in files_in_day_path and not os.path.exists(day_output_path / "segment3"):
                os.makedirs(day_output_path / "segment3")

In [ ]:
# Main driver function
def assign_affected_and_less_affected(interpolated_dir: Path, output_dir: Path):
    """
    Reads subject clinical info (affected side) and reorganizes each subject's
    interpolated sensor data into standardized columns:
      - affected finger
      - affected wrist
      - less affected finger
      - less affected wrist
    It also computes:
      - affected finger - wrist
      - less affected finger - wrist
    Then saves each result as assigned.csv in a mirrored folder structure.
    """

    # Load the demographics CSV containing "SubjID" and "Affected Side"
    df_demo = pd.read_csv("path/to/clinical_scores.csv")

    # Get all subjects and their day folder paths
    valid_subject_day_paths = get_subject_day_paths(data_path=interpolated_dir)
    create_dirs(valid_subject_day_paths=valid_subject_day_paths, output_dir=output_dir)

    for subject, day_paths in valid_subject_day_paths.items():
        # Look up affected side (left/right) for this subject
        affected_side = df_demo.loc[df_demo["SubjID"] == subject, "Affected Side"].iloc[0]

        for day_path in day_paths:
            print(subject, day_path.name[:6])
            files_in_day_path = os.listdir(day_path)

            # --- Case 1: Single interpolated.csv (no segments) ---
            if "segment1" not in files_in_day_path:
                interpolated_day_dict = read_interpolated_sensor_csv(day_path)

                # Reassign signals based on affected side
                if affected_side == "left":
                    assigned_day_dict = {
                        "affected finger": interpolated_day_dict["left finger"],
                        "affected wrist": interpolated_day_dict["left wrist"],
                        "less affected finger": interpolated_day_dict["right finger"],
                        "less affected wrist": interpolated_day_dict["right wrist"],
                    }
                else:  # affected_side == "right"
                    assigned_day_dict = {
                        "affected finger": interpolated_day_dict["right finger"],
                        "affected wrist": interpolated_day_dict["right wrist"],
                        "less affected finger": interpolated_day_dict["left finger"],
                        "less affected wrist": interpolated_day_dict["left wrist"],
                    }

                # Compute difference features
                df_assigned_day = pd.DataFrame({
                    "affected finger": assigned_day_dict["affected finger"],
                    "affected wrist": assigned_day_dict["affected wrist"],
                    "less affected finger": assigned_day_dict["less affected finger"],
                    "less affected wrist": assigned_day_dict["less affected wrist"],
                    "affected finger - wrist": assigned_day_dict["affected finger"] - assigned_day_dict["affected wrist"],
                    "less affected finger - wrist": assigned_day_dict["less affected finger"] - assigned_day_dict["less affected wrist"],
                })

                # Save result
                df_assigned_day.to_csv(output_dir / subject / day_path.name / "assigned.csv", index=False)

            # --- Case 2: Day has multiple segment subfolders ---
            else:
                segment_paths = list(day_path.glob("segment*"))
                for segment_path in segment_paths:
                    interpolated_day_dict = read_interpolated_sensor_csv(segment_path)

                    if affected_side == "left":
                        assigned_day_dict = {
                            "affected finger": interpolated_day_dict["left finger"],
                            "affected wrist": interpolated_day_dict["left wrist"],
                            "less affected finger": interpolated_day_dict["right finger"],
                            "less affected wrist": interpolated_day_dict["right wrist"],
                        }
                    else:
                        assigned_day_dict = {
                            "affected finger": interpolated_day_dict["right finger"],
                            "affected wrist": interpolated_day_dict["right wrist"],
                            "less affected finger": interpolated_day_dict["left finger"],
                            "less affected wrist": interpolated_day_dict["left wrist"],
                        }

                    df_assigned_day = pd.DataFrame({
                        "affected finger": assigned_day_dict["affected finger"],
                        "affected wrist": assigned_day_dict["affected wrist"],
                        "less affected finger": assigned_day_dict["less affected finger"],
                        "less affected wrist": assigned_day_dict["less affected wrist"],
                        "affected finger - wrist": assigned_day_dict["affected finger"] - assigned_day_dict["affected wrist"],
                        "less affected finger - wrist": assigned_day_dict["less affected finger"] - assigned_day_dict["less affected wrist"],
                    })

                    # Save assigned CSV per segment
                    df_assigned_day.to_csv(output_dir / subject / day_path.name / segment_path.name / "assigned.csv", index=False)



In [ ]:
assign_affected_and_less_affected(interpolated_dir=Path("path/to/your/data"),
                                  output_dir=Path("path/to/your/data"))
